# Dark Pattern Recognition - Full Colab Notebook

This notebook trains the project models and runs the demo workflow in Google Colab:

- Stage 1: binary dark pattern classifier
- Stage 2: dark pattern type/category classifier
- Demo filters for reducing obvious false positives
- Text analysis examples
- Webpage scan examples
- Figures and output files for the project report/page


## 1. Install Dependencies

Run this once at the start of a Colab session.

In [ ]:
!pip -q install pandas scikit-learn joblib matplotlib seaborn requests beautifulsoup4 lxml


## 2. Set Up Files

Expected folder layout:

```text
ai4all-20c-darkpatterns/
  datasets/
    krishuppal - dark-patterns.csv
    adarshm09 - dark-pattern-dataset.csv
    devitachi - dark-pattern.csv
    mohitsharma527 - dark-patterns-on-ecommerce-platforms.csv
    - pattern_classifications.csv
```

If you are using Google Drive, mount Drive and set `PROJECT_DIR` to your project folder.

In [ ]:
from pathlib import Path

# Optional in Colab:
# from google.colab import drive
# drive.mount('/content/drive')
# PROJECT_DIR = Path('/content/drive/MyDrive/Code/ai4all-20c-darkpatterns')

# Default: assume the notebook is running from the project folder.
PROJECT_DIR = Path.cwd()
DATASET_DIR = PROJECT_DIR / 'datasets'
OUTPUT_DIR = PROJECT_DIR / 'outputs'
FIGURE_DIR = PROJECT_DIR / 'reports' / 'figures'
ARTIFACT_DIR = PROJECT_DIR / 'artifacts'

for folder in [OUTPUT_DIR, FIGURE_DIR, ARTIFACT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print('Project folder:', PROJECT_DIR)
print('Dataset folder exists:', DATASET_DIR.exists())


## 3. Imports and Constants

In [ ]:
import math
import re
import warnings
from dataclasses import dataclass
from typing import Optional

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns
from bs4 import BeautifulSoup
from sklearn.base import BaseEstimator
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

RANDOM_STATE = 42
TEXT_COLUMN = 'text'
LABEL_COLUMN = 'label'
CATEGORY_COLUMN = 'category'
SOURCE_COLUMN = 'source'
NOT_DARK_PATTERN = 'Not Dark Pattern'

PRIMARY_BINARY_FILE = 'krishuppal - dark-patterns.csv'
ADARSH_CATEGORY_FILE = 'adarshm09 - dark-pattern-dataset.csv'
DEVITACHI_FILE = 'devitachi - dark-pattern.csv'
MOHIT_FILE = 'mohitsharma527 - dark-patterns-on-ecommerce-platforms.csv'
AKASH_CLASSIFICATION_FILE = '- pattern_classifications.csv'


## 4. Dataset Loading Code

In [ ]:
def dataset_path(filename: str) -> Path:
    path = DATASET_DIR / filename
    if not path.exists():
        raise FileNotFoundError(f'Dataset not found: {path}')
    return path


def normalize_text(value) -> str:
    if pd.isna(value):
        return ''
    return ' '.join(str(value).strip().split())


def _drop_empty_text(df: pd.DataFrame) -> pd.DataFrame:
    return df[df[TEXT_COLUMN].str.len() > 0].reset_index(drop=True)


def deduplicate_text(df: pd.DataFrame) -> pd.DataFrame:
    return df.drop_duplicates(subset=[TEXT_COLUMN]).reset_index(drop=True)


def deduplicate_text_with_source_history(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, group in df.groupby(TEXT_COLUMN, sort=False):
        row = group.iloc[0].copy()
        row[SOURCE_COLUMN] = '; '.join(sorted(set(group[SOURCE_COLUMN].astype(str))))
        rows.append(row)
    return pd.DataFrame(rows).reset_index(drop=True)


def load_primary_binary_dataset() -> pd.DataFrame:
    raw = pd.read_csv(dataset_path(PRIMARY_BINARY_FILE))
    df = pd.DataFrame({
        TEXT_COLUMN: raw['text'].map(normalize_text),
        LABEL_COLUMN: raw['label'].astype(int),
        CATEGORY_COLUMN: raw['Pattern Category'].map(normalize_text),
        SOURCE_COLUMN: 'krishuppal',
    })
    return _drop_empty_text(df)


def load_adarsh_category_dataset() -> pd.DataFrame:
    raw = pd.read_csv(dataset_path(ADARSH_CATEGORY_FILE))
    category = raw['Category'].map(normalize_text)
    df = pd.DataFrame({
        TEXT_COLUMN: raw['Title'].map(normalize_text),
        LABEL_COLUMN: (category != NOT_DARK_PATTERN).astype(int),
        CATEGORY_COLUMN: category,
        SOURCE_COLUMN: 'adarshm09',
    })
    return _drop_empty_text(df)


def load_ecommerce_patterns_dataset(filename: str, source_name: str) -> pd.DataFrame:
    raw = pd.read_csv(dataset_path(filename))
    category = raw['Pattern Category'].map(normalize_text)
    df = pd.DataFrame({
        TEXT_COLUMN: raw['Pattern String'].map(normalize_text),
        LABEL_COLUMN: (category != NOT_DARK_PATTERN).astype(int),
        CATEGORY_COLUMN: category,
        'pattern_type': raw['Pattern Type'].map(normalize_text),
        'page_location': raw['Where in website?'].map(normalize_text),
        'deceptive_flag': raw['Deceptive?'].map(normalize_text),
        SOURCE_COLUMN: source_name,
    })
    return _drop_empty_text(df)


def load_devitachi_binary_dataset() -> pd.DataFrame:
    df = load_ecommerce_patterns_dataset(DEVITACHI_FILE, 'devitachi')
    df[LABEL_COLUMN] = 1
    return df


def load_akash_binary_dataset() -> pd.DataFrame:
    raw = pd.read_csv(dataset_path(AKASH_CLASSIFICATION_FILE))
    raw = raw.dropna(subset=['Pattern String', 'classification']).copy()
    raw['classification'] = raw['classification'].astype(int)
    category = raw['classification'].map({0: NOT_DARK_PATTERN, 1: 'Dark Pattern'})
    df = pd.DataFrame({
        TEXT_COLUMN: raw['Pattern String'].map(normalize_text),
        LABEL_COLUMN: raw['classification'],
        CATEGORY_COLUMN: category,
        SOURCE_COLUMN: 'akashnath29',
    })
    return _drop_empty_text(df)


def load_expanded_binary_dataset() -> pd.DataFrame:
    frames = [
        load_primary_binary_dataset(),
        load_devitachi_binary_dataset()[[TEXT_COLUMN, LABEL_COLUMN, CATEGORY_COLUMN, SOURCE_COLUMN]],
        load_akash_binary_dataset(),
    ]
    return deduplicate_text_with_source_history(pd.concat(frames, ignore_index=True, sort=False))


def load_secondary_category_datasets() -> pd.DataFrame:
    frames = [
        load_adarsh_category_dataset(),
        load_devitachi_binary_dataset(),
        load_ecommerce_patterns_dataset(MOHIT_FILE, 'mohitsharma527'),
    ]
    return deduplicate_text(pd.concat(frames, ignore_index=True, sort=False))


def load_dark_pattern_category_dataset() -> pd.DataFrame:
    df = load_secondary_category_datasets()
    df = df[df[LABEL_COLUMN] == 1].copy()
    df = df[df[CATEGORY_COLUMN] != NOT_DARK_PATTERN].copy()
    return df.reset_index(drop=True)


def validate_binary_dataset(df: pd.DataFrame) -> None:
    required = {TEXT_COLUMN, LABEL_COLUMN, CATEGORY_COLUMN, SOURCE_COLUMN}
    missing = required.difference(df.columns)
    if missing:
        raise ValueError(f'Missing columns: {sorted(missing)}')
    if set(df[LABEL_COLUMN].unique()) != {0, 1}:
        raise ValueError('Binary labels must be exactly 0 and 1')
    if df[TEXT_COLUMN].isna().any() or (df[TEXT_COLUMN].str.len() == 0).any():
        raise ValueError('Dataset contains empty text values')


def validate_category_dataset(df: pd.DataFrame) -> None:
    required = {TEXT_COLUMN, CATEGORY_COLUMN, SOURCE_COLUMN}
    missing = required.difference(df.columns)
    if missing:
        raise ValueError(f'Missing columns: {sorted(missing)}')
    if df[CATEGORY_COLUMN].nunique() < 2:
        raise ValueError('Category dataset needs at least two categories')


## 5. Model Training Code

In [ ]:
@dataclass(frozen=True)
class ModelResult:
    name: str
    pipeline: Pipeline
    accuracy: float
    precision: float
    recall: float
    f1: float

    def as_dict(self):
        return {
            'model': self.name,
            'accuracy': self.accuracy,
            'precision': self.precision,
            'recall': self.recall,
            'f1': self.f1,
        }


@dataclass(frozen=True)
class CategoryModelResult:
    name: str
    pipeline: Pipeline
    accuracy: float
    precision_macro: float
    recall_macro: float
    f1_macro: float

    def as_dict(self):
        return {
            'model': self.name,
            'accuracy': self.accuracy,
            'precision_macro': self.precision_macro,
            'recall_macro': self.recall_macro,
            'f1_macro': self.f1_macro,
        }


def make_classifier(model_name: str) -> BaseEstimator:
    if model_name == 'Logistic Regression':
        return LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
    if model_name == 'Naive Bayes':
        return MultinomialNB()
    if model_name == 'Linear SVM':
        return LinearSVC(random_state=RANDOM_STATE)
    if model_name == 'Decision Tree':
        return DecisionTreeClassifier(max_depth=25, random_state=RANDOM_STATE)
    if model_name == 'Random Forest':
        return RandomForestClassifier(n_estimators=150, max_depth=35, random_state=RANDOM_STATE, n_jobs=-1)
    raise ValueError(f'Unknown model name: {model_name}')


def make_pipeline(model_name: str) -> Pipeline:
    return Pipeline([
        ('tfidf', TfidfVectorizer(
            lowercase=True,
            ngram_range=(1, 2),
            min_df=2,
            max_features=8000,
            stop_words='english',
        )),
        ('classifier', make_classifier(model_name)),
    ])


def model_names():
    return ['Logistic Regression', 'Naive Bayes', 'Linear SVM', 'Decision Tree', 'Random Forest']


def train_and_compare_binary(df: pd.DataFrame) -> list[ModelResult]:
    x_train, x_test, y_train, y_test = train_test_split(
        df[TEXT_COLUMN], df[LABEL_COLUMN], test_size=0.2, stratify=df[LABEL_COLUMN], random_state=RANDOM_STATE
    )
    results = []
    for name in model_names():
        pipeline = make_pipeline(name)
        pipeline.fit(x_train, y_train)
        predictions = pipeline.predict(x_test)
        results.append(ModelResult(
            name=name,
            pipeline=pipeline,
            accuracy=accuracy_score(y_test, predictions),
            precision=precision_score(y_test, predictions, zero_division=0),
            recall=recall_score(y_test, predictions, zero_division=0),
            f1=f1_score(y_test, predictions, zero_division=0),
        ))
    return sorted(results, key=lambda result: result.f1, reverse=True)


def train_and_compare_categories(df: pd.DataFrame) -> list[CategoryModelResult]:
    x_train, x_test, y_train, y_test = train_test_split(
        df[TEXT_COLUMN], df[CATEGORY_COLUMN], test_size=0.2, stratify=df[CATEGORY_COLUMN], random_state=RANDOM_STATE
    )
    results = []
    for name in model_names():
        pipeline = make_pipeline(name)
        pipeline.fit(x_train, y_train)
        predictions = pipeline.predict(x_test)
        results.append(CategoryModelResult(
            name=name,
            pipeline=pipeline,
            accuracy=accuracy_score(y_test, predictions),
            precision_macro=precision_score(y_test, predictions, average='macro', zero_division=0),
            recall_macro=recall_score(y_test, predictions, average='macro', zero_division=0),
            f1_macro=f1_score(y_test, predictions, average='macro', zero_division=0),
        ))
    return sorted(results, key=lambda result: result.f1_macro, reverse=True)


def results_to_frame(results):
    return pd.DataFrame([result.as_dict() for result in results])


## 6. Demo Filters and Type Helpers

In [ ]:
PRICE_OR_DISCOUNT_RE = re.compile(r'(\$\s?\d|\d+\s?%+\s*off|sale price|previous price|was\s+\$|now\s+\$)', re.I)
PRESSURE_LANGUAGE_RE = re.compile(r'(hurry|last chance|limited time|limited offer|ends soon|ends tonight|sale ends|deal ends|expires|almost gone|selling fast|only\s+\d+\s+(left|remaining)|\d+\s+(left|remaining)\s+in stock|low stock|while supplies last|act now)', re.I)
URGENCY_RE = re.compile(r'(hurry|last chance|limited time|limited offer|ends soon|ends tonight|sale ends|deal ends|expires|act now)', re.I)
SCARCITY_RE = re.compile(r'(almost gone|selling fast|only\s+\d+\s+(left|remaining)|\d+\s+(left|remaining)\s+in stock|low stock|while supplies last)', re.I)
SOCIAL_PROOF_RE = re.compile(r'(\d+[\d,]*\s+(people|customers|users|shoppers|visitors|members)\b|people are (looking|viewing|watching)|someone just bought|popular|trending|best[- ]?seller|reviews?|ratings?|deals bought)', re.I)
CONFIRMSHAMING_RE = re.compile(r"(no thanks|i don't want|i do not want|not interested).*(save|deal|offer|discount|money)", re.I)
PRODUCT_TITLE_NOISE_RE = re.compile(r'(\b(us|usa|uk|eu)\s+stock\b|\bnear\s+mint\b|\bopen\s+box\b|\brefurbished\b|\b\d+(\.\d+)?\s?(mm|gb|tb|mah|hz|inch|inches|mp|w)\b|\b\d+\s?x\s?\d+\b|\bf/\d+(\.\d+)?\b|\b(android|ios)\s+\d+\b)', re.I)
PRODUCT_CATALOG_WORD_RE = re.compile(r'(tablet|phone|camera|lens|telephoto|zoom|touchscreen|processor|storage|battery|screen|display|laptop|monitor|headphones|speaker|watch)', re.I)
BARE_SOCIAL_COUNTER_RE = re.compile(r'^(deals bought|customers|users|reviews|ratings|downloads|students|members)\s*:\s*[\d,]+$', re.I)
TESTIMONIAL_FRAGMENT_RE = re.compile(r"^(thanks\s+[a-z][a-z .'-]*!?|thank you\s+[a-z][a-z .'-]*!?|why\s+i\s+bought\s+this!?|showed\s+me\s+.+|.+\s+walk\s+through\s+call\s+yesterday\.?)$", re.I)
CONTENT_TITLE_RE = re.compile(r'^(\d+\)?\s*)?(copy|write|create|generate|build|structure|template|lesson|module)\b', re.I)


def contains_pressure_language(snippet: str) -> bool:
    return bool(PRESSURE_LANGUAGE_RE.search(snippet))


def infer_dark_pattern_type(snippet: str) -> str:
    matches = []
    if URGENCY_RE.search(snippet):
        matches.append('Urgency')
    if SCARCITY_RE.search(snippet):
        matches.append('Scarcity')
    if SOCIAL_PROOF_RE.search(snippet):
        matches.append('Social Proof')
    if CONFIRMSHAMING_RE.search(snippet):
        matches.append('Confirmshaming')
    return ' + '.join(matches) if matches else 'Unclear from text alone'


def is_simple_price_or_discount_snippet(snippet: str) -> bool:
    return bool(PRICE_OR_DISCOUNT_RE.search(snippet)) and not contains_pressure_language(snippet)


def is_low_context_product_snippet(snippet: str) -> bool:
    if contains_pressure_language(snippet):
        return False
    has_catalog_word = bool(PRODUCT_CATALOG_WORD_RE.search(snippet))
    has_noise = bool(PRODUCT_TITLE_NOISE_RE.search(snippet))
    has_many_numbers = sum(char.isdigit() for char in snippet) >= 6
    return has_noise and (has_catalog_word or has_many_numbers)


def is_low_context_web_snippet(snippet: str) -> bool:
    if contains_pressure_language(snippet):
        return False
    cleaned = ' '.join(snippet.split())
    words = cleaned.split()
    if len(words) <= 3:
        return True
    if BARE_SOCIAL_COUNTER_RE.match(cleaned):
        return True
    if TESTIMONIAL_FRAGMENT_RE.match(cleaned):
        return True
    if CONTENT_TITLE_RE.match(cleaned) and len(words) <= 12:
        return True
    return False


## 7. Train Both Models

In [ ]:
primary_df = load_primary_binary_dataset()
validate_binary_dataset(primary_df)

category_df = load_dark_pattern_category_dataset()
validate_category_dataset(category_df)

print('Binary rows:', len(primary_df))
print(primary_df[LABEL_COLUMN].value_counts().sort_index())
print('\nCategory rows:', len(category_df))
print(category_df[CATEGORY_COLUMN].value_counts().sort_index())


In [ ]:
binary_results = train_and_compare_binary(primary_df)
binary_metrics = results_to_frame(binary_results)
binary_metrics.to_csv(OUTPUT_DIR / 'colab_primary_model_metrics.csv', index=False)
binary_metrics.to_csv(PROJECT_DIR / 'reports' / 'model_metrics.csv', index=False)

best_binary = binary_results[0]
joblib.dump(best_binary.pipeline, ARTIFACT_DIR / 'best_binary_model.joblib')

print('Best binary model:', best_binary.name)
display(binary_metrics)


In [ ]:
category_results = train_and_compare_categories(category_df)
category_metrics = results_to_frame(category_results)
category_metrics.to_csv(OUTPUT_DIR / 'colab_category_model_metrics.csv', index=False)
category_metrics.to_csv(PROJECT_DIR / 'reports' / 'category_model_metrics.csv', index=False)

best_category = category_results[0]
joblib.dump(best_category.pipeline, ARTIFACT_DIR / 'best_category_model.joblib')

print('Best category model:', best_category.name)
display(category_metrics)


## 8. Generate Figures

In [ ]:
def save_current_figure(filename: str):
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    path = FIGURE_DIR / filename
    plt.tight_layout()
    plt.savefig(path, dpi=180, bbox_inches='tight')
    print('Saved:', path)


plt.figure(figsize=(8, 5))
sns.barplot(data=binary_metrics, x='f1', y='model', color='#ff4b4b')
plt.xlim(0, 1)
plt.title('Binary Model Comparison by F1 Score')
plt.xlabel('F1 Score')
plt.ylabel('Model')
save_current_figure('model_comparison.png')
plt.show()

plt.figure(figsize=(8, 5))
sns.barplot(data=category_metrics, x='f1_macro', y='model', color='#2f80ed')
plt.xlim(0, 1)
plt.title('Category Model Comparison by Macro F1')
plt.xlabel('Macro F1 Score')
plt.ylabel('Model')
save_current_figure('category_model_comparison.png')
plt.show()

plt.figure(figsize=(8, 5))
category_counts = category_df[CATEGORY_COLUMN].value_counts().sort_values(ascending=True)
sns.barplot(x=category_counts.values, y=category_counts.index, color='#00a676')
plt.title('Dark Pattern Category Distribution')
plt.xlabel('Rows')
plt.ylabel('Category')
save_current_figure('category_distribution.png')
plt.show()


## 9. Prediction Helpers

In [ ]:
def confidence_for_prediction(model: Pipeline, text: str, predicted_label) -> Optional[float]:
    if hasattr(model, 'predict_proba'):
        classes = list(model.classes_)
        probabilities = model.predict_proba([text])[0]
        return float(probabilities[classes.index(predicted_label)])
    if hasattr(model, 'decision_function'):
        scores = np.asarray(model.decision_function([text]))
        if scores.ndim == 1 and len(scores) == 1:
            value = float(scores[0])
            confidence = 1 / (1 + math.exp(-abs(value)))
            return confidence
        if scores.ndim == 2:
            scores = scores[0]
            shifted = scores - scores.max()
            probs = np.exp(shifted) / np.exp(shifted).sum()
            classes = list(model.classes_)
            return float(probs[classes.index(predicted_label)])
    return None


def predict_binary(text: str, model: Pipeline = None):
    model = model or best_binary.pipeline
    predicted = int(model.predict([text])[0])
    confidence = confidence_for_prediction(model, text, predicted)
    label_name = 'Dark Pattern' if predicted == 1 else 'Not Dark Pattern'
    return label_name, confidence


def predict_category(text: str, model: Pipeline = None):
    model = model or best_category.pipeline
    predicted = model.predict([text])[0]
    confidence = confidence_for_prediction(model, text, predicted)
    return predicted, confidence


def should_filter_text(text: str, for_web_scan: bool = False) -> tuple[bool, str]:
    if is_simple_price_or_discount_snippet(text):
        return True, 'Filtered: plain price or discount without pressure language'
    if is_low_context_product_snippet(text):
        return True, 'Filtered: catalog/product-spec text without pressure language'
    if for_web_scan and is_low_context_web_snippet(text):
        return True, 'Filtered: vague short snippet, bare counter, or testimonial fragment'
    return False, ''


def analyze_text(text: str, apply_filters: bool = True):
    filtered, filter_reason = should_filter_text(text, for_web_scan=False) if apply_filters else (False, '')
    label, confidence = predict_binary(text)
    if filtered:
        label = 'Not Dark Pattern'
    category, category_confidence = ('Not applicable', None)
    if label == 'Dark Pattern':
        category, category_confidence = predict_category(text)
    return {
        'text': text,
        'main_answer': label,
        'binary_confidence': confidence,
        'possible_type': category if category != 'Not applicable' else infer_dark_pattern_type(text),
        'type_confidence': category_confidence,
        'filter_note': filter_reason,
    }


def show_analysis(result: dict):
    confidence = result['binary_confidence']
    confidence_text = 'n/a' if confidence is None else f'{confidence:.1%}'
    type_conf = result['type_confidence']
    type_text = result['possible_type'] if type_conf is None else f"{result['possible_type']} ({type_conf:.1%})"
    print('Text:', result['text'])
    print('Main answer:', result['main_answer'])
    print('Model confidence:', confidence_text)
    print('Possible type:', type_text)
    if result['filter_note']:
        print(result['filter_note'])


## 10. Try Text Examples

In [ ]:
examples = {
    'Urgency example': 'Hurry, this limited time offer ends tonight!',
    'Scarcity example': 'Only 2 left in stock. Order now before it sells out.',
    'Social proof example': '1,243 people are looking at this item right now.',
    'Filter example': 'Previous price: $99.00 47% off',
    'Catalog title example': 'US stock 64GB touchscreen tablet refurbished open box',
    'Neutral example': 'This cotton shirt is available in blue and black.',
}

for name, text in examples.items():
    print('\n' + '=' * 80)
    print(name)
    show_analysis(analyze_text(text, apply_filters=True))


## 11. Webpage Scanner

This Colab version uses `requests` and `BeautifulSoup` by default. It is faster and simpler than Playwright, but it may miss text that only appears after JavaScript runs.

In [ ]:
def extract_visible_text_with_requests(url: str, timeout: int = 20) -> str:
    headers = {
        'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/125 Safari/537.36'
    }
    response = requests.get(url, headers=headers, timeout=timeout)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, 'lxml')
    for tag in soup(['script', 'style', 'noscript', 'svg', 'iframe']):
        tag.decompose()
    return soup.get_text('\n')


def split_visible_text(page_text: str, min_chars: int = 4, max_chars: int = 220) -> list[str]:
    parts = []
    for line in page_text.splitlines():
        cleaned = ' '.join(line.split())
        if min_chars <= len(cleaned) <= max_chars:
            parts.append(cleaned)
    seen = set()
    unique = []
    for item in parts:
        key = item.lower()
        if key not in seen:
            seen.add(key)
            unique.append(item)
    return unique


def score_snippets(snippets: list[str], threshold: float = 0.65, apply_filters: bool = True, limit: int = 10):
    rows = []
    for snippet in snippets:
        filtered, filter_note = should_filter_text(snippet, for_web_scan=True) if apply_filters else (False, '')
        label, confidence = predict_binary(snippet)
        if filtered:
            continue
        if label != 'Dark Pattern':
            continue
        if confidence is not None and confidence < threshold:
            continue
        category, category_confidence = predict_category(snippet)
        rows.append({
            'snippet': snippet,
            'prediction': label,
            'confidence': confidence,
            'possible_type': category,
            'type_confidence': category_confidence,
        })
    rows = sorted(rows, key=lambda row: row['confidence'] or 0, reverse=True)
    return rows[:limit]


def scan_webpage(url: str, threshold: float = 0.65, limit: int = 10, apply_filters: bool = True):
    print('Scanning:', url)
    page_text = extract_visible_text_with_requests(url)
    snippets = split_visible_text(page_text)
    results = score_snippets(snippets, threshold=threshold, apply_filters=apply_filters, limit=limit)
    print('Visible snippets reviewed:', len(snippets))
    print('Results shown:', len(results))
    return pd.DataFrame(results)


In [ ]:
web_examples = [
    'https://www.ulta.com/promotion/sale',
    'https://toonbee.ai/',
    'https://www.cnn.com/',
]

# Change the URL or threshold here.
scan_results = scan_webpage(web_examples[0], threshold=0.65, limit=10, apply_filters=True)
display(scan_results)


## 12. Optional Playwright Scanner for JavaScript-Heavy Pages

Use this only when the simple scanner misses text because the site renders content with JavaScript. It is slower and may need a fresh Colab runtime.

In [ ]:
# Optional setup:
# !pip -q install playwright
# !playwright install chromium

# Optional usage:
# from playwright.async_api import async_playwright
#
# async def extract_visible_text_with_playwright_colab(url: str, wait_ms: int = 3000) -> str:
#     async with async_playwright() as p:
#         browser = await p.chromium.launch(headless=True)
#         page = await browser.new_page(viewport={'width': 1365, 'height': 900})
#         await page.goto(url, wait_until='domcontentloaded', timeout=45000)
#         await page.wait_for_timeout(wait_ms)
#         text = await page.locator('body').inner_text(timeout=10000)
#         await browser.close()
#         return text
#
# page_text = await extract_visible_text_with_playwright_colab('https://toonbee.ai/')
# snippets = split_visible_text(page_text)
# playwright_results = pd.DataFrame(score_snippets(snippets, threshold=0.65, limit=10, apply_filters=True))
# display(playwright_results)


## 13. Notes for Presentation

- The first model answers whether text looks like a dark pattern.
- The second model predicts a likely dark pattern category only after the first model flags text as suspicious.
- The category model is helpful for explanations, but it is not perfect because category labels are smaller and noisier than the binary labels.
- Demo filters are not part of the model. They reduce obvious false positives such as plain discounts, product spec titles, bare counters, and short testimonial fragments.
- Webpage scanning depends on what text can be extracted from the page. Dynamic pages, blocked requests, popups, and repeated snippets can affect results.
